In [13]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

ROOT = Path.cwd().parent
sys.path.append(str(ROOT / "src"))



In [14]:
from faster_whisper import WhisperModel

model = WhisperModel("small", device="cpu", compute_type="int8")

segments, info = model.transcribe(
    str(ROOT / "data" / "audio" / "rec_201.wav"),
    language="pl",
    beam_size=5,
    vad_filter=True,
    word_timestamps=True
)

for seg in segments:
    print(f"[{seg.start:.1f}-{seg.end:.1f}] {seg.text}")

[0.1-7.7]  No dobra, najlepiej pamiętam taką sytuację z jednego projektu i komersowego.
[8.4-12.6]  Zrobiliśmy dplay w piątek po południu, co już samo w sobie.
[13.2-16.0]  No nie było najlepszą praktyką.
[16.4-21.3]  Po jakichś dwudziestu minutach zaczęli pisać ludzie z obsługi klienta,
[21.7-25.1]  że klienci mogą po prostu dokończyć się, dokończyć swojego zamówienia.
[25.9-30.1]  Wszedłem do grafany i od razu było widać, że liczba błędów na płatnościach
[30.1-34.6]  poszła strasznie do góry, praktycznie zaraz po wdrożeniu.
[35.0-37.7]  Nie chcieliśmy szukać przyczyny, bo klienci nie mogli płacić.
[37.8-42.8]  Wiadomą, że to jest krytyczny moduł, więc od razu zrobiliśmy rollback do poprzedniej wersji
[42.8-45.1]  i sytuacja się, powiedzmy, uspokoiła.
[45.9-48.0]  Później na spokojnie, jak to przeanalizowaliśmy,
[48.8-52.7]  to okazało się, że po prostu problem był w jednej zmianie w walidacji adresu.
[52.7-56.8]  No to przygotowaliśmy kotfiks, sprawdziliśmy go już na testowym środowi

In [15]:
# Wariant 1: czysty Whisper, konfiguracja
import csv, json, time
from faster_whisper import WhisperModel

MODEL_NAME = "large-v3-turbo"
PARAMS = {"language": "pl", "beam_size": 5, "vad_filter": True, "word_timestamps": True}
OUT_DIR = ROOT / "results" / "asr" / "whisper"
OUT_DIR.mkdir(parents=True, exist_ok=True)

with open(ROOT / "data" / "recordings.csv", encoding="utf-8") as f:
    recordings = list(csv.DictReader(f))

model = WhisperModel(MODEL_NAME, device="cpu", compute_type="int8")

In [16]:
def transcribe_file(file_id, variant="whisper", params=PARAMS):
    start_time = time.perf_counter()
    segments, info = model.transcribe(str(ROOT / "data" / "audio" / f"{file_id}.wav"), **params)

    segs = []
    for seg in segments:  # dopiero tutaj Whisper naprawdę transkrybuje
        segs.append({
            "start": seg.start, "end": seg.end, "text": seg.text,
            "no_speech_prob": seg.no_speech_prob,
            "avg_logprob": seg.avg_logprob,
            "compression_ratio": seg.compression_ratio,  # wysoka wartość = zapętlenie
            "words": [{"word": w.word, "start": w.start, "end": w.end, "probability": w.probability}
                      for w in seg.words or []],
        })

    return {
        "id": file_id,
        "variant": variant,
        "model": MODEL_NAME,
        "params": params,
        "duration": info.duration,
        "processing_time": time.perf_counter() - start_time,
        "segments": segs,
    }

In [17]:
# Transkrypcja wszystkich nagrań; gotowe pliki są pomijane, żeby przeliczyć od nowa, usuń JSON
def run_variant(variant, params, out_dir):
    out_dir.mkdir(parents=True, exist_ok=True)
    for rec in recordings:
        file_id = rec["file_id"]
        out_path = out_dir / f"{file_id}.json"
        if out_path.exists():
            continue
        print(f"{file_id}…", end=" ", flush=True)
        result = transcribe_file(file_id, variant, params)
        out_path.write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding="utf-8")
        print(f"{result['duration']:.0f} s audio w {result['processing_time']:.0f} s")

run_variant("whisper", PARAMS, OUT_DIR)

In [18]:
# Filtr halucynacji, stosowany do zapisanych wyników (zmiana reguły nie wymaga ponownej transkrypcji)
KNOWN_HALLUCINATIONS = ("za oglądanie", "napisy wykonane", "napisy stworzone", "subskrybuj")
MAX_WORDS_PER_SECOND = 8  # ludzie mówią 2–4 słowa/s, 8 jest fizycznie niemożliwe

def is_hallucination(seg):
    text = seg["text"].strip().lower()
    duration = max(seg["end"] - seg["start"], 0.01)
    too_fast = len(text.split()) / duration > MAX_WORDS_PER_SECOND
    return too_fast or any(p in text for p in KNOWN_HALLUCINATIONS)

def load_result(file_id, out_dir=OUT_DIR):
    return json.loads((out_dir / f"{file_id}.json").read_text(encoding="utf-8"))

def clean_text(result):
    return " ".join(s["text"].strip() for s in result["segments"] if not is_hallucination(s))

In [19]:
dev_ids = [rec["file_id"] for rec in recordings if rec["split"] == "dev"]

def preview(out_dir=OUT_DIR):
    for file_id in dev_ids:
        result = load_result(file_id, out_dir)
        dropped = [s["text"].strip() for s in result["segments"] if is_hallucination(s)]
        print(f"## {file_id} ({result['duration']:.0f} s), odrzucone: {dropped or 'brak'}")
        print(clean_text(result), "\n")

preview()

## rec_201 (79 s), odrzucone: ['Dziękuje za oglądanie.']
No dobra, najlepiej pamiętam taką sytuację. Z jednego projektu e-commerce'owego zrobiliśmy deploy w piątek po południu, co już samo w sobie nie było najlepszą praktyką. Po jakichś 20 minutach zaczęli pisać ludzie z obsługi klienta, że klienci nie mogą po prostu dokończyć swojego zamówienia. Wszedłem do Grafany i od razu było widać, że liczba błędów na płatnościach poszła strasznie do góry, praktycznie zaraz po wdrożeniu. Nie chcieliśmy szukać przyczyny, no bo klienci nie mogli płacić, a wiadomo, że to jest krytyczny moduł, więc od razu zrobiliśmy rollback do poprzedniej wersji i sytuacja się, powiedzmy, że uspokoiła. No później na spokojnie jak to przeanalizowaliśmy, no to okazało się, że po prostu problem był w jednej zmianie w walidacji adresu. No to przygotowaliśmy kod fixa, sprawdziliśmy go już na testowym środowisku i wypuściliśmy w poniedziałek rano. No i cała ta sytuacja dała nam po prostu dwie rzeczy, dała nam do myślenia

### Wariant z przerywnikami (yyy)

Whisper domyślnie „wygładza” mowę: pomija `yyy`, powtórzenia i poprawianie się. Z podpowiedzią (`initial_prompt`) napisaną dosłownie część z nich zostaje. Jeśli na całym `dev` WER spadnie, a terminy się nie pogorszą, ta podpowiedź wchodzi do ustawień bazowych wariantów 2–4.

In [20]:
# Ten sam Whisper, ale z podpowiedzią napisaną dosłownie, z przerywnikami; model naśladuje jej styl
VERBATIM_PROMPT = "Yyy, no więc, eee, pracowałem nad tym, mmm, projektem i, yyy, no, to było, eee, dość trudne."
VERBATIM_PARAMS = {**PARAMS, "initial_prompt": VERBATIM_PROMPT}
VERBATIM_DIR = ROOT / "results" / "asr" / "whisper_verbatim"

run_variant("whisper_verbatim", VERBATIM_PARAMS, VERBATIM_DIR)

In [21]:
preview(VERBATIM_DIR)

## rec_201 (79 s), odrzucone: brak
Yyy, no dobra, no, najlepiej pamiętam taką sytuację, z jednego projektu e-commerce'owego, zrobiliśmy deploy w piątek po południu, co już samo w sobie, no nie było najlepszą, lepszą praktyką, po jakichś dwudziestu minutach zaczęli pisać ludzie z obsługi klienta, że klienci nie mogą po prostu dokończyć, dokończyć swojego zamówienia, no to wszedłem do Grafany i, i od razu było widać, że liczba błędów na płatnościach, poszła strasznie do góry, yyy, praktycznie zaraz po wdrożeniu. Nie chcieliśmy szukać przyczyny, no bo klienci nie mogli płacić, a wiadomo, że to jest krytyczny moduł, więc, więc od razu zrobiliśmy rollback do poprzedniej wersji, no i sytuacja się, no powiedzmy, że uspokoiła. Yyy, później na spokojenie, jak to przeanalizowaliśmy, no to okazało się, że po prostu problem był w jednej zmianie w walidacji adresu. No to przygotowaliśmy kod fixa, sprawdziliśmy go już na testowym środowisku i wypuściliśmy w poniedziałek rano. No i cała ta sytuacja d

### Porównanie wariantów na dev

WER i CER są liczone bez przerywników. Termin jest trafiony, gdy Whisper zapisał w tym samym miejscu dokładnie tę samą formę co wzorzec; zmyślony, gdy pojawia się tam, gdzie we wzorcu terminu nie było (`src/metrics.py`).

In [22]:
# Porównanie na dev: przerywniki, WER, CER i terminy IT
import jiwer
import pandas as pd
from transcripts import count_fillers, load_term_names, load_transcript, normalize, parse_transcript, plain_text
from metrics import build_lexicon, prf, term_counts

# formy terminów ze wszystkich nagrań; słownik należy do narzędzia oceny, a nie do systemu, więc test może tu być
forms = [t["form"] for rec in recordings for t in parse_transcript(load_transcript(rec["file_id"]))[1]]
lexicon = build_lexicon(load_term_names(), forms)

prompt_echo = " ".join(normalize(VERBATIM_PROMPT))
rows, refs, base_hyps, verb_hyps = [], [], [], []
totals = {"base": {"tp": 0, "fn": 0, "fp": 0}, "verbatim": {"tp": 0, "fn": 0, "fp": 0}}
for file_id in dev_ids:
    gold = load_transcript(file_id)
    words, terms = parse_transcript(gold)
    base = clean_text(load_result(file_id))
    verb = clean_text(load_result(file_id, VERBATIM_DIR))
    base_words, verb_words = normalize(base), normalize(verb)
    refs.append(" ".join(words))
    base_hyps.append(" ".join(base_words))
    verb_hyps.append(" ".join(verb_words))

    counts = {"base": term_counts(words, terms, base_words, lexicon),
              "verbatim": term_counts(words, terms, verb_words, lexicon)}
    for variant, c in counts.items():
        for key, value in c.items():
            totals[variant][key] += value

    rows.append({
        "id": file_id,
        "fillers_gold": count_fillers(plain_text(gold)),
        "fillers_base": count_fillers(base),
        "fillers_verbatim": count_fillers(verb),
        "wer_base": jiwer.wer(refs[-1], base_hyps[-1]),
        "wer_verbatim": jiwer.wer(refs[-1], verb_hyps[-1]),
        "cer_base": jiwer.cer(refs[-1], base_hyps[-1]),
        "cer_verbatim": jiwer.cer(refs[-1], verb_hyps[-1]),
        "terms": len(terms),
        "f1_base": prf(**counts["base"])[2],  # None, gdy nagranie nie ma terminów
        "f1_verbatim": prf(**counts["verbatim"])[2],
        "prompt_echo": prompt_echo in verb_hyps[-1],  # Whisper czasem przepisuje samą podpowiedź
    })

df = pd.DataFrame(rows).set_index("id")
print(f"WER bez przerywników na całym dev: base {jiwer.wer(refs, base_hyps):.1%} | verbatim {jiwer.wer(refs, verb_hyps):.1%}")
print(f"CER bez przerywników na całym dev: base {jiwer.cer(refs, base_hyps):.1%} | verbatim {jiwer.cer(refs, verb_hyps):.1%}")
print(f"Przerywniki na całym dev: wzorzec {df.fillers_gold.sum()}, base {df.fillers_base.sum()}, verbatim {df.fillers_verbatim.sum()}")
# najpierw sumy ze wszystkich nagrań, potem jeden wynik, tak jak przy WER
for variant, c in totals.items():
    precision, recall, f1 = prf(**c)
    print(f"Terminy na całym dev ({variant}): precision {precision:.1%} | recall {recall:.1%} | F1 {f1:.1%} "
          f"({c['tp']} trafionych, {c['fn']} pominiętych, {c['fp']} zmyślonych)")

percent_columns = ["wer_base", "wer_verbatim", "cer_base", "cer_verbatim", "f1_base", "f1_verbatim"]
df.style.format({col: "{:.1%}" for col in percent_columns}, na_rep="–")

WER bez przerywników na całym dev: base 8.9% | verbatim 6.4%
CER bez przerywników na całym dev: base 5.0% | verbatim 3.1%
Przerywniki na całym dev: wzorzec 97, base 1, verbatim 18
Terminy na całym dev (base): precision 96.6% | recall 77.8% | F1 86.2% (56 trafionych, 16 pominiętych, 2 zmyślonych)
Terminy na całym dev (verbatim): precision 98.2% | recall 76.4% | F1 85.9% (55 trafionych, 17 pominiętych, 1 zmyślonych)


,fillers_gold,fillers_base,fillers_verbatim,wer_base,wer_verbatim,cer_base,cer_verbatim,terms,f1_base,f1_verbatim,prompt_echo
id,,,,,,,,,,,
rec_201,15,0,3,10.5%,3.7%,6.6%,1.2%,7,92.3%,92.3%,False
rec_203,7,0,1,6.0%,6.5%,2.3%,3.1%,5,100.0%,88.9%,False
rec_101,10,0,6,4.6%,3.7%,1.1%,0.9%,8,93.3%,100.0%,False
rec_103,11,0,0,11.2%,10.5%,4.7%,4.6%,15,75.0%,75.0%,False
rec_001,21,1,1,8.3%,7.6%,5.6%,4.7%,19,82.4%,81.3%,False
rec_003,23,0,7,8.4%,5.7%,5.7%,3.5%,18,88.2%,88.2%,False
rec_005,10,0,0,15.6%,9.1%,9.5%,3.7%,0,–,–,False


In [23]:
# Pominięte terminy: forma ze wzorca → co Whisper zapisał w tym miejscu (∅ = nic); podpowiedź do hotwords
from metrics import align

def missed_terms(file_id, out_dir=OUT_DIR):
    words, terms = parse_transcript(load_transcript(file_id))
    hyp = normalize(clean_text(load_result(file_id, out_dir)))
    ref_equal, _, hyp_to_ref = align(words, hyp)
    ref_to_hyp = {i: j for j, i in enumerate(hyp_to_ref) if i is not None}
    missed = []
    for t in terms:
        if not all(ref_equal[t["start"]:t["end"]]):
            written = " ".join(hyp[ref_to_hyp[i]] for i in range(t["start"], t["end"]) if i in ref_to_hyp)
            missed.append(f"{t['form']} → {written or '∅'}")
    return missed

for file_id in dev_ids:
    print(f"## {file_id}")
    print("  base:    ", ", ".join(missed_terms(file_id)) or "brak")
    print("  verbatim:", ", ".join(missed_terms(file_id, VERBATIM_DIR)) or "brak")

## rec_201
  base:     hotfixa → fixa
  verbatim: hotfixa → fixa
## rec_203
  base:     brak
  verbatim: poc → poca
## rec_101
  base:     mergujemy → mergeujemy
  verbatim: brak
## rec_103
  base:     vocare → wokare, vocare → wokare, postgresql → sql, pgvector → vector, cacheowania → keszowania, kafki → kawki
  verbatim: vocare → wokare, vocare → wokare, postgresql → sql, pgvector → vector, cacheowania → keszowania, kafki → kawki
## rec_001
  base:     ragowego → reagowego, featury → ∅, c → csharpie, hangfire → hackfirea, vercelu → ∅
  verbatim: ragowego → reagowego, featury → featurey, c → csharpie, pgvector → pgvektor, hangfire → hackfirea, vercelu → patreska
## rec_003
  base:     brancha → brancza, merge requeście → merge jest, mergowane → mergeowane
  verbatim: brancha → brancza, merge requeście → merge jest, mergowane → mergeowane
## rec_005
  base:     brak
  verbatim: brak
